# Problem Book P1, Chapter 9: Integrals with Arithmetic Inside (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: four experiments that push the
problems further, with exercises first and solutions at the end. Only `numpy`, `sympy`, `mpmath` and `matplotlib` are needed.

*Before running a cell, predict its output.*

In [ ]:
import math, random, itertools
from math import comb, factorial, lcm, log, sqrt, pi
from fractions import Fraction as F
import numpy as np
import sympy as sp
import mpmath as mp
import matplotlib.pyplot as plt
mp.mp.dps = 30
random.seed(1); rng = np.random.default_rng(1)
x, y = sp.symbols('x y')

def legendre(n):                       # coefficients of P_n(x) = (1/n!) D^n x^n (1-x)^n
    return [(-1)**k * comb(n, k) * comb(n + k, k) for k in range(n + 1)]
def d(n):
    r = 1
    for k in range(1, n + 1): r = lcm(r, k)
    return r
print([legendre(n) for n in range(4)], [d(n) for n in range(1, 11)])

## Part A: checks

**Problem 9.1 ($\log 2$).** Rodrigues's formula, $d_nI_n=a_n+b_n\log2$ with integers, and $0<I_n\le(3-2\sqrt2)^n$.

In [ ]:
for n in (1, 2, 3, 6, 10):
    P = sp.expand(sp.diff(x**n * (1 - x)**n, x, n) / sp.factorial(n))
    assert sp.Poly(P, x).all_coeffs()[::-1] == legendre(n)
    I = sp.integrate(P / (1 + x), (x, 0, 1))
    b = sp.expand(d(n) * I).coeff(sp.log(2)); a = sp.expand(d(n) * I - b * sp.log(2))
    In = mp.quad(lambda t: t**n * (1 - t)**n / (1 + t)**(n + 1), [0, 1])
    print(n, a, b, b == d(n) * sum(c * (-1)**k for k, c in enumerate(legendre(n))), mp.nstr(In, 8), In <= (3 - 2 * mp.sqrt(2))**n)

**Problem 9.2 (powers of $e$).** The integer $M_n=aF(1)-bF(0)$ would have to lie in $]0,1[$ for large $n$: watch the bound fall, here for $k=3$
and a hypothetical denominator $b=10^6$.

In [ ]:
k, b = 3, 10**6
print([(n, float(b * k * math.e**k * (k * k / 4)**n / factorial(n))) for n in (10, 20, 30, 40)])

**Problem 9.3 (double integrals).** The closed forms against numerical double integrals.

In [ ]:
def J(r, s, w):                         # (rational part, coefficient of zeta(w+1)); w=1: kernel 1/(1-xy), w=2: -log(xy)/(1-xy)
    if r < s: r, s = s, r
    if r > s: return (sum(F(1, k**w) for k in range(s + 1, r + 1)) / (r - s), 0)
    return (-w * sum(F(1, k**(w + 1)) for k in range(1, r + 1)), w)
for (r, s) in ((0, 0), (2, 1), (3, 3), (4, 1)):
    q1 = mp.quad(lambda u, v: u**r * v**s / (1 - u * v), [0, 1], [0, 1])
    q2 = mp.quad(lambda u, v: -mp.log(u * v) * u**r * v**s / (1 - u * v), [0, 1], [0, 1])
    a1, b1 = J(r, s, 1); a2, b2 = J(r, s, 2)
    print((r, s), mp.nstr(q1 - (a1 + b1 * mp.zeta(2)), 3), mp.nstr(q2 - (a2 + b2 * mp.zeta(3)), 3))
print(mp.quad(lambda u, v: 1 / (1 + u * v), [0, 1], [0, 1]), mp.pi**2 / 12)

**Problems 9.4 and 9.5 (Beukers).** Exact $a_n,b_n$, the integrality of $d_n^2J_n$ and $d_n^3K_n$, the integral formulas for small $n$, and the
bounds of parts (c) and (d).

In [ ]:
def beukers(n, w):
    p = legendre(n); q = p if w == 2 else [(-1)**k * comb(n, k) for k in range(n + 1)]
    a = sum(p[r] * q[s] * J(r, s, w)[0] for r in range(n + 1) for s in range(n + 1))
    b = sum(p[r] * q[s] * J(r, s, w)[1] for r in range(n + 1) for s in range(n + 1))
    return a, b
for n in range(1, 9):
    a2, b2 = beukers(n, 1); a3, b3 = beukers(n, 2)
    print(n, (a2 * d(n)**2).denominator == 1, b2, (a3 * d(n)**3).denominator == 1, b3)
for n in (1, 2):
    a2, b2 = beukers(n, 1); a3, b3 = beukers(n, 2)
    print(mp.nstr(a2 + b2 * mp.zeta(2), 12), mp.nstr((-1)**n * mp.quad(lambda u, v: (u*(1-u)*v*(1-v))**n / (1-u*v)**(n+1), [0, 1], [0, 1]), 12))
    print(mp.nstr(a3 + b3 * mp.zeta(3), 12), mp.nstr(mp.quad(lambda u, v, t: (u*(1-u)*v*(1-v)*t*(1-t))**n / (1-(1-u*v)*t)**(n+1), [0, 1], [0, 1], [0, 1]), 12))
tau = (mp.sqrt(5) - 1) / 2
print(mp.nstr(tau**5, 6), mp.nstr((mp.sqrt(2) - 1)**4, 6), mp.nstr(9 * tau**5, 4), mp.nstr(27 * (mp.sqrt(2) - 1)**4, 4))

The coefficients $b_n$ for $\zeta(2)$ are $3,19,147,1251,\dots$ and those for $\zeta(3)$ are twice the Apéry numbers $5,73,1445,33001,\dots$ A brute-force
search for the maximum of the integrand of 9.5 over a grid confirms $(\sqrt2-1)^4$:

In [ ]:
g = np.linspace(0.001, 0.999, 70); X, Y, W = np.meshgrid(g, g, g, indexing='ij')
H = X * (1 - X) * Y * (1 - Y) * W * (1 - W) / (1 - (1 - X * Y) * W)
i = np.unravel_index(np.argmax(H), H.shape)
print(H.max(), float((mp.sqrt(2) - 1)**4), g[i[0]], g[i[1]], g[i[2]], 1 / sqrt(2), 1 / (1 + 1 / sqrt(2)))

**Problem 9.6 (Sondow).** The two double integrals.

In [ ]:
print(mp.quad(lambda u, v: (1 - u) / ((1 - u * v) * -mp.log(u * v)), [0, 1], [0, 1]), mp.euler)
print(mp.quad(lambda u, v: (1 - u) / ((1 + u * v) * -mp.log(u * v)), [0, 1], [0, 1]), mp.log(4 / mp.pi))

**Problem 9.7 ($\pi<22/7$).** Exact integrals and the approximations $-4(-4)^{-n}r_n$.

In [ ]:
for n in range(1, 5):
    I = sp.integrate(x**(4*n) * (1 - x)**(4*n) / (1 + x**2), (x, 0, 1))
    r = sp.nsimplify(sp.expand(I - (-4)**n * sp.pi / 4))
    approx = -4 * sp.Rational(-4)**(-n) * r
    print(n, r if n == 1 else '...', float(abs(sp.pi - approx)), 4.0**(1 - 5 * n))
print(sp.Rational(1, 1260) < sp.Rational(22, 7) - sp.pi < sp.Rational(1, 630))

**Problems 9.8–9.10 (central binomial sums).**

In [ ]:
S = lambda f, a=1: mp.nsum(f, [a, mp.inf])
print(S(lambda n: 1 / (n * mp.binomial(2*n, n))), mp.pi / (3 * mp.sqrt(3)))
print(S(lambda n: 1 / mp.binomial(2*n, n), 0), mp.mpf(4) / 3 + 2 * mp.pi * mp.sqrt(3) / 27)
print(S(lambda n: 1 / (n**2 * mp.binomial(2*n, n))), mp.pi**2 / 18)
c = lambda n: mp.binomial(2*n, n) / 4**n
N = 10**5; print(mp.fsum(4**n / (n**2 * mp.binomial(2*n, n)) for n in range(1, 2001)) + 2 * mp.sqrt(mp.pi / 2000), mp.pi**2 / 2)
print(mp.fsum(c(n) / (2*n + 1) for n in range(0, 20001)) + 1 / mp.sqrt(mp.pi * 20000), mp.pi / 2)
print(S(lambda n: c(n) / n), 2 * mp.log(2))
print([mp.nstr(4 * M * c(M)**2, 8) for M in (10, 1000, 10**5)], mp.nstr(4 / mp.pi, 8),
      mp.nstr(mp.fsum(c(n)**2 / (n + 1) for n in range(1000)), 8), mp.nstr(4 * 1000 * c(1000)**2, 8))
print(S(lambda n: 1 / (n**4 * mp.binomial(2*n, n))) * 36 / 17, mp.zeta(4), "  Comtet")
print(S(lambda n: (-1)**(n - 1) / (n**3 * mp.binomial(2*n, n))) * 5 / 2, mp.zeta(3), "  Apéry")

(The two slowly convergent series were summed with a tail correction $\sum_{n>N}\approx2\sqrt{\pi/N}$, respectively $\approx1/\sqrt{\pi N}$, from $c_n\sim1/\sqrt{\pi n}$.)

**Problem 9.11 (Catalan moments).**

In [ ]:
for n in range(6):
    m = sp.integrate(x**(2*n) * sp.sqrt(4 - x**2), (x, -2, 2)) / (2 * sp.pi)
    print(n, sp.simplify(m), sp.catalan(n))
print([mp.nstr(mp.mpf(int(sp.catalan(n))) / (mp.mpf(4)**n / (mp.sqrt(mp.pi) * mp.mpf(n)**1.5)), 5) for n in (10, 100, 1000)])

**Problem 9.12 (integrals of $\{1/x\}$).** Exact sums over $[k,k+1]$, accelerated.

In [ ]:
f1 = mp.nsum(lambda k: mp.log(1 + 1/k) - 1/(k + 1), [1, mp.inf])
f2 = mp.nsum(lambda k: 1 - 2*k*mp.log(1 + 1/k) + k/(k + 1), [1, mp.inf])
f3 = mp.nsum(lambda k: (1/k) * (1/k - 1/(k + 1)), [1, mp.inf])
print(f1, 1 - mp.euler); print(f2, mp.log(2*mp.pi) - mp.euler - 1); print(f3, mp.zeta(2) - 1)

**Problem 9.13 (remainders on average).** $\frac1x\sum_{n\le x}\{x/n\}$ and $\frac1{N^2}\sum(N\bmod n)$.

In [ ]:
for X in (10**3, 10**5, 10**7):
    n = np.arange(1, X + 1); fr = (X / n) % 1
    print(X, round(fr.mean(), 5), round(float(1 - mp.euler), 5), round(float((X % n).sum()) / X**2, 5), round(1 - pi**2 / 12, 5),
          round((fr.sum() - (1 - float(mp.euler)) * X) / sqrt(X), 3))
print(2 * mp.nsum(lambda k: 1/(2*k) - 1/(2*(k + 1)) - 1/(2*(k + 1)**2), [1, mp.inf]), 2 - mp.zeta(2))

**Problem 9.14 (the law of $\{1/U\}$).** Simulation against the density $g$, and the two probabilities.

In [ ]:
U = rng.random(10**6); V = 1 / U
print(round(np.mean(np.floor(V) % 2 == 1), 4), round(log(2), 4), round(np.mean(V % 1 < 0.5), 4), round(2 - 2 * log(2), 4),
      round(np.mean(V % 1), 4), round(float(1 - mp.euler), 4), round(np.var(V % 1), 4), round(float(mp.log(2*mp.pi) - 2 + mp.euler - mp.euler**2), 4))
X = 10**7; n = np.arange(1, X + 1); print(np.mean((X // n) % 2 == 1))
print([round(float(mp.psi(1, 1 + u)), 4) for u in (0, 0.5, 1)], round(pi**2 / 6, 4), round(pi**2 / 6 - 1, 4))

**Problem 9.15 (remainders modulo primes).** The convergence is slow, as the proof suggests.

In [ ]:
P = np.array(list(sp.primerange(2, 2 * 10**7)), dtype=np.int64)
for X in (10**4, 10**5, 10**6, 10**7, 2 * 10**7 - 1):
    p = P[P <= X]; print(X, round(float(((X / p) % 1).mean()), 4))
print(round(float(1 - mp.euler), 4))

**Problem 9.16 (waiting to pass 1).**

In [ ]:
def first_pass(M=10**5):
    out = []
    for _ in range(M):
        s, n = 0.0, 0
        while s <= 1: s += random.random(); n += 1
        out.append(n)
    return np.array(out)
Nn = first_pass(); print(Nn.mean(), math.e, Nn.var(), 3 * math.e - math.e**2)
L = []
for _ in range(10**5):
    a, n = random.random(), 1
    while True:
        b = random.random()
        if b < a: break
        a, n = b, n + 1
    L.append(n)
print(np.mean(L) + 1, [round(np.mean(Nn > k), 4) for k in range(5)], [round(1 / factorial(k), 4) for k in range(5)])

**Problem 9.17 (Eulerian numbers).** Formula against brute-force descent counts, and $P(\lfloor S_n\rfloor=k)$ by simulation.

In [ ]:
A = lambda n, k: sum((-1)**j * comb(n + 1, j) * (k + 1 - j)**n for j in range(k + 1))
for n in range(1, 8):
    cnt = [0] * n
    for s in itertools.permutations(range(n)): cnt[sum(s[i] > s[i + 1] for i in range(n - 1))] += 1
    assert cnt == [A(n, k) for k in range(n)]
print([A(5, k) for k in range(5)])
S5 = rng.random((10**6, 5)).sum(1); print(np.bincount(S5.astype(int)) / 10**6, [A(5, k) / 120 for k in range(5)])

**Problem 9.18 (powers of $\sin t/t$).** The finite formula, numerical integration, and $\sqrt{6\pi/n}$.

In [ ]:
def sinc_int(n): return mp.pi / (2**(n - 1) * mp.factorial(n - 1)) * mp.fsum((-1)**k * comb(n, k) * (n - 2*k)**(n - 1) for k in range(0, (n + 1) // 2))
print([mp.nstr(sinc_int(n) / mp.pi, 8) for n in (2, 3, 4)])
for n in (5, 6):
    q = 2 * mp.quadosc(lambda t: (mp.sin(t) / t)**n if t else 1, [0, mp.inf], period=2 * mp.pi)
    print(n, mp.nstr(q, 12), mp.nstr(sinc_int(n), 12))
print([mp.nstr(sinc_int(n) / mp.sqrt(6 * mp.pi / n), 6) for n in (10, 100, 400)])

**Problem 9.19 (Borwein).** The exact deficit from the simplex in part (c), and from a general formula: for $Y=\sum_ka_kV_k$,
$P(Y\le y)=\frac1{m!\prod 2a_k}\sum_{\varepsilon\in\{\pm1\}^m}\bigl(\prod\varepsilon_k\bigr)\bigl(y+\sum\varepsilon_ka_k\bigr)_+^m$ (inclusion–exclusion, as in 9.17).

In [ ]:
def tail(a, y=1):                       # exact P(|Y| > y) for Y = sum a_k V_k, V_k uniform on [-1, 1], a_k rational
    m = len(a); tot = F(0)
    for eps in itertools.product((1, -1), repeat=m):
        z = y + sum(e * ak for e, ak in zip(eps, a))
        if z > 0: tot += math.prod(eps) * z**m
    cdf = tot / (factorial(m) * math.prod(2 * ak for ak in a))
    return 2 * (1 - cdf)
a = [F(1, 2*k + 1) for k in range(1, 8)]
delta = sum(a) - 1
simplex = mp.pi * mp.mpf(delta.numerator)**7 / mp.mpf(delta.denominator)**7 * math.prod(2*k + 1 for k in range(1, 8)) / (2**7 * factorial(7))
print(delta, mp.nstr(simplex, 6), mp.nstr(mp.pi / 2 * mp.mpf(tail(a).numerator) / tail(a).denominator, 6))
print([float(tail([F(1, 2*k + 1) for k in range(1, m + 1)])) for m in range(5, 9)])

**Problem 9.20 (breaking a stick).**

In [ ]:
for n in (3, 4, 6):
    cuts = np.sort(rng.random((2 * 10**5, n - 1)), axis=1)
    L = np.diff(np.concatenate([np.zeros((len(cuts), 1)), cuts, np.ones((len(cuts), 1))], axis=1), axis=1)
    T = rng.random(len(L)); idx = (cuts < T[:, None]).sum(1)
    print(n, round(np.mean(L.max(1) < 0.5), 4), 1 - n / 2**(n - 1), round(L[np.arange(len(L)), idx].mean(), 4), round(2 / (n + 1), 4))

**Problem 9.21 (longest and shortest piece).**

In [ ]:
for n in (5, 20, 100):
    cuts = np.sort(rng.random((10**5, n - 1)), axis=1)
    L = np.diff(np.concatenate([np.zeros((len(cuts), 1)), cuts, np.ones((len(cuts), 1))], axis=1), axis=1)
    H = sum(1 / k for k in range(1, n + 1))
    print(n, round(L.max(1).mean() * n / H, 4), round(L.min(1).mean() * n * n, 4))

**Problem 9.22 (sophomore's dream).**

In [ ]:
print(mp.quad(lambda t: t**-t, [0, 1]), mp.fsum(mp.mpf(n)**-n for n in range(1, 11)))
print(mp.quad(lambda t: t**t, [0, 1]), mp.fsum((-1)**(n + 1) * mp.mpf(n)**-n for n in range(1, 30)))
print([float(sum(mp.mpf(n)**-n for n in range(N + 1, 40))) for N in (9, 10)])

## Part B: laboratory

**L9.1 (Apéry's approximations).** Compute the Apéry numbers $u_n$ from the recurrence $n^3u_n=(34n^3-51n^2+27n-5)u_{n-1}-(n-1)^3u_{n-2}$,
$u_0=1$, $u_1=5$, and the companion sequence $v_n$ with the same recurrence, $v_0=0$, $v_1=6$. How many correct digits of $\zeta(3)$
does $v_n/u_n$ give, as a function of $n$? Compare the number of digits gained per step with $\log_{10}(1+\sqrt2)^8$.

**L9.2 (Borwein integrals at larger $m$).** With the exact tail formula of 9.19, compute the deficit $\frac\pi2P(|Y|\ge1)$ for $a_k=\frac1{2k+1}$,
$m=7,\dots,11$, and for the sequence $a_k=\frac1{k+1}$, where the pattern breaks as soon as $\frac12+\cdots+\frac1{m+1}>1$. Plot $\log_{10}$ of the deficits.

**L9.3 (remainders, more finely).** For $x=10^7$, histogram $\{x/n\}$ over $n\le x$ and $\{x/p\}$ over primes $p\le x$, and compare both with the
density $g$ of Problem 9.14. Which one fits better, and why?

**L9.4 (descents are normal).** For $n=20$, compare the Eulerian distribution $A(20,k)/20!$ with the normal law of mean $\frac{n-1}2$ and
variance $\frac{n+1}{12}$, and with the empirical law of $\lfloor S_{20}\rfloor$.

## Solutions

### L9.1

In [ ]:
mp.mp.dps = 300
u, v = [1, 5], [0, 6]
for n in range(2, 31):
    c = 34*n**3 - 51*n**2 + 27*n - 5
    u.append((c * u[-1] - (n - 1)**3 * u[-2]) // n**3 if (c * u[-1] - (n - 1)**3 * u[-2]) % n**3 == 0 else F(c * u[-1] - (n - 1)**3 * u[-2], n**3))
    v.append(F(c * v[-1] - (n - 1)**3 * v[-2], n**3))
print(u[:7], all(isinstance(t, int) for t in u))
digs = [float(-mp.log10(abs(mp.zeta(3) - mp.mpf(F(v[n]).numerator) / F(v[n]).denominator / u[n]))) for n in range(1, 31)]
print([round(t, 1) for t in digs[::5]], round((digs[-1] - digs[9]) / 20, 3), round(float(mp.log10((1 + mp.sqrt(2))**8)), 3))
mp.mp.dps = 30

The Apéry numbers are integers (a fact that is not obvious from the recurrence, which divides by $n^3$), and each step gains about
$\log_{10}(1+\sqrt2)^8=3.06$ digits: $u_n$ grows like $(1+\sqrt2)^{4n}$ and the error $\zeta(3)-v_n/u_n$ is about $u_n^{-2}$. The
denominators of $v_n$ divide $2d_n^3$, about $e^{3n}$, which is why the irrationality proof works: with $q_n=2d_n^3u_n$,
$q_n|\zeta(3)-v_n/u_n|$ behaves like $\bigl(e^3/(1+\sqrt2)^4\bigr)^n=0.59^n\to0$.

### L9.2

In [ ]:
defs = []
for m in range(7, 12):
    t = tail([F(1, 2*k + 1) for k in range(1, m + 1)]); defs.append(float(mp.pi / 2 * t))
print([f"{t:.3e}" for t in defs])
print([f"{float(mp.pi / 2 * tail([F(1, k + 1) for k in range(1, m + 1)])):.3e}" for m in range(1, 6)])
plt.figure(figsize=(5, 3)); plt.plot(range(7, 12), np.log10(defs), "o-"); plt.xlabel("m"); plt.ylabel("log10 deficit"); plt.show()

For $a_k=\frac1{2k+1}$ the deficit grows from $2.3\cdot10^{-11}$ at $m=7$ to visible sizes within a few steps, since $\sum a_k$ grows
like $\frac12\log m$. For $a_k=\frac1{k+1}$ the pattern breaks at $m=3$ ($\frac12+\frac13+\frac14>1$), and the deficit is then already about $10^{-3}$. A pattern that holds seven times, and fails by $10^{-11}$, is a warning about numerical evidence.

### L9.3

In [ ]:
X = 10**7; n = np.arange(1, X + 1); p = P[P <= X]
u = np.linspace(0, 1, 201); gu = np.array([float(mp.psi(1, 1 + t)) for t in u])
plt.figure(figsize=(6, 3))
plt.hist((X / n) % 1, bins=50, density=True, alpha=0.5, label="n <= x")
plt.hist((X / p) % 1, bins=50, density=True, alpha=0.5, label="primes p <= x")
plt.plot(u, gu, "k", label="g(u)"); plt.legend(); plt.show()
hn, _ = np.histogram((X / n) % 1, bins=50, range=(0, 1), density=True)
hp, _ = np.histogram((X / p) % 1, bins=50, range=(0, 1), density=True)
mid = np.linspace(0.01, 0.99, 50); gm = np.array([float(mp.psi(1, 1 + t)) for t in mid])
print(round(np.abs(hn - gm).max(), 3), round(np.abs(hp - gm).max(), 3))

Both fit; the integers fit much better. For $n\le x$ the error is controlled by the divisor problem, of size about $x^{-3/4}$ in
relative terms, while for primes it is controlled by the error in the prime number theorem relative to $\frac x{\log x}$, and
by the fluctuations of a sample of only $\pi(10^7)=664\,579$ points.

### L9.4

In [ ]:
n = 20
Aeul = np.array([A(n, k) for k in range(n)], dtype=float) / factorial(n)
k = np.arange(n); mu, var = (n - 1) / 2, (n + 1) / 12
normal = np.exp(-(k - mu)**2 / (2 * var)) / np.sqrt(2 * np.pi * var)
emp = np.bincount(rng.random((2 * 10**5, n)).sum(1).astype(int), minlength=n) / 2e5
print(round(abs(Aeul.sum() - 1), 12), round((Aeul * k).sum(), 6), round((Aeul * (k - mu)**2).sum(), 6), var)
plt.figure(figsize=(6, 3)); plt.bar(k, Aeul, alpha=0.5, label="A(20,k)/20!"); plt.plot(k, normal, "r.-", label="normal")
plt.plot(k, emp, "k+", label="floor(S_20)"); plt.legend(); plt.show()

Mean $\frac{19}2$ and variance $\frac{21}{12}=1.75$ exactly, and the normal curve is already very close: the number of descents of a
random permutation is the integer part of a sum of $n$ uniforms, so the central limit theorem applies to it directly (with mean
$\frac n2$ for $S_n$, shifted by $\frac12$ for the integer part).